In [2]:
# ============================================================
# Google Drive 연결 및 프로젝트 루트 설정
# ============================================================

from google.colab import drive
import os

drive.mount("/content/drive")

PROJECT_ROOT = (
    "/content/drive/MyDrive/"
    "코드잇_파트2_3팀_프로젝트/"
    "pill-object-detection"
)

os.chdir(PROJECT_ROOT)

print("현재 작업 디렉터리:", os.getcwd())

Mounted at /content/drive
현재 작업 디렉터리: /content/drive/MyDrive/코드잇_파트2_3팀_프로젝트/pill-object-detection


In [30]:
# ============================================================
# Import
# ============================================================

from pathlib import Path

import torch
import pandas as pd

from src.inference import (
    inference_faster_rcnn,
    predictions_to_submission,
    save_submission,
)

In [4]:
# ============================================================
# Device
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("사용 device:", device)

사용 device: cuda


In [27]:
# ============================================================
# Path
# ============================================================

DATASET_ROOT = (
    Path(PROJECT_ROOT)
    / "data"
    / "dataset"
    / "cleaning_data"
    / "sprint_ai_project1_data_260809_baseline_dataset"
)

TEST_IMAGE_DIR = DATASET_ROOT / "test_images"

CHECKPOINT_PATH = (
    Path(PROJECT_ROOT)
    / "outputs"
    / "checkpoints"
    / "faster_rcnn_best.pth"
)

SUBMISSION_PATH = (
    Path(PROJECT_ROOT)
    / "outputs"
    / "submissions"
    / "faster_rcnn_baseline_thr005.csv"
)

print("test image:", TEST_IMAGE_DIR)
print("checkpoint:", CHECKPOINT_PATH)
print("submission:", SUBMISSION_PATH)

test image: /content/drive/MyDrive/코드잇_파트2_3팀_프로젝트/pill-object-detection/data/dataset/cleaning_data/sprint_ai_project1_data_260809_baseline_dataset/test_images
checkpoint: /content/drive/MyDrive/코드잇_파트2_3팀_프로젝트/pill-object-detection/outputs/checkpoints/faster_rcnn_best.pth
submission: /content/drive/MyDrive/코드잇_파트2_3팀_프로젝트/pill-object-detection/outputs/submissions/faster_rcnn_baseline_thr005.csv


In [6]:
# ============================================================
# Path 확인
# ============================================================

print("test image dir exists:", TEST_IMAGE_DIR.exists())
print("checkpoint exists:", CHECKPOINT_PATH.exists())
print("submission dir exists:", SUBMISSION_PATH.parent.exists())

test image dir exists: True
checkpoint exists: True
submission dir exists: True


In [7]:
# ============================================================
# Test Image 확인
# ============================================================

image_paths = sorted(
    list(TEST_IMAGE_DIR.glob("*.png"))
    + list(TEST_IMAGE_DIR.glob("*.jpg"))
    + list(TEST_IMAGE_DIR.glob("*.jpeg"))
)

print("test image 수:", len(image_paths))

if image_paths:
    print("첫 번째 이미지:", image_paths[0].name)
    print("마지막 이미지:", image_paths[-1].name)

test image 수: 842
첫 번째 이미지: 1.png
마지막 이미지: 999.png


In [8]:
# ============================================================
# Faster R-CNN 모델 생성
# ============================================================

from torchvision.models.detection import (
    FasterRCNN_ResNet50_FPN_Weights,
    fasterrcnn_resnet50_fpn,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor


NUM_CLASSES = 57

model = fasterrcnn_resnet50_fpn(
    weights=None,
)

in_features = model.roi_heads.box_predictor.cls_score.in_features

model.roi_heads.box_predictor = FastRCNNPredictor(
    in_features,
    NUM_CLASSES,
)

model = model.to(device)

print("Faster R-CNN 모델 생성 완료")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 227MB/s]


Faster R-CNN 모델 생성 완료


In [9]:
# ============================================================
# Checkpoint 확인
# ============================================================

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
)

print("checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("checkpoint keys:")
    for key in checkpoint.keys():
        print("-", key)

checkpoint type: <class 'dict'>
checkpoint keys:
- epoch
- model_state_dict
- optimizer_state_dict
- val_map
- scheduler_state_dict


In [10]:
# ============================================================
# Checkpoint Load
# ============================================================

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to(device)
model.eval()

print("checkpoint epoch:", checkpoint["epoch"])
print("validation mAP:", checkpoint["val_map"])
print("Faster R-CNN checkpoint 로드 완료")

checkpoint epoch: 19
validation mAP: 0.7740236520767212
Faster R-CNN checkpoint 로드 완료


In [11]:
# ============================================================
# Single Image Inference Test
# ============================================================

test_predictions = inference_faster_rcnn(
    model=model,
    image_paths=image_paths[:1],
    device=device,
)

print("prediction 수:", len(test_predictions))
print("image_id:", test_predictions[0]["image_id"])
print("box 수:", len(test_predictions[0]["boxes"]))
print("label 수:", len(test_predictions[0]["labels"]))
print("score 수:", len(test_predictions[0]["scores"]))

prediction 수: 1
image_id: 1
box 수: 31
label 수: 31
score 수: 31


In [12]:
print("boxes:", test_predictions[0]["boxes"][:3])
print("labels:", test_predictions[0]["labels"][:3])
print("scores:", test_predictions[0]["scores"][:3])

boxes: [[148.30340576171875, 255.44378662109375, 374.218994140625, 378.1698303222656], [568.1218872070312, 121.49139404296875, 897.1823120117188, 466.2629699707031], [575.0219116210938, 96.0262222290039, 921.7951049804688, 446.59490966796875]]
labels: [53, 51, 19]
scores: [0.9519889950752258, 0.3390964865684509, 0.33670932054519653]


In [13]:
# ============================================================
# 전체 Test Image Inference
# ============================================================

predictions = inference_faster_rcnn(
    model=model,
    image_paths=image_paths,
    device=device,
)

print("전체 prediction 수:", len(predictions))

전체 prediction 수: 842


In [20]:
# ============================================================
# 원본 category_id <-> 학습 label mapping 생성
# ============================================================

import json


def build_category_mapping(annotation_dir: Path):
    category_ids = set()
    category_names = {}

    if not annotation_dir.exists():
        print(
            f"[경고] train annotation 폴더를 찾을 수 없습니다: "
            f"{annotation_dir}"
        )
        return {}, {}

    # 하위 디렉터리까지 재귀적으로 JSON 탐색
    for json_path in sorted(annotation_dir.rglob("*.json")):
        with json_path.open("r", encoding="utf-8") as file:
            data = json.load(file)

        for category in data.get("categories", []):
            category_id = int(category["id"])

            category_ids.add(category_id)
            category_names[category_id] = category.get(
                "name",
                "",
            )

    sorted_category_ids = sorted(category_ids)

    label_to_category_id = {
        label: category_id
        for label, category_id in enumerate(
            sorted_category_ids,
            start=1,
        )
    }

    category_id_to_label = {
        category_id: label
        for label, category_id
        in label_to_category_id.items()
    }

    return (
        label_to_category_id,
        category_id_to_label,
    )

In [22]:
# ============================================================
# 학습 Dataset 기반 Class Mapping 복원
# ============================================================

from src.PillDetectionDataset import PillDetectionDataset

mapping_dataset = PillDetectionDataset(
    root=DATASET_ROOT,
    transforms=None,
    image_dir_name="train_images",
    annotation_dir_name="train_annotations",
    label_offset=1,
    strict=False,
    validate_image_size=True,
)

/content/drive/MyDrive/코드잇_파트2_3팀_프로젝트/pill-object-detection/src/PillDetectionDataset.py:268: RuntimeWarning: 이미지 K-003351-020014-020238_0_2_0_2_75_000_200.png의 알약 ID 수는 3개지만 대응 JSON은 0개입니다.
  self._handle_problem(


In [23]:
# ============================================================
# label -> 원본 category_id mapping
# ============================================================

label_to_category_id = dict(mapping_dataset.label2cat)

print(
    "label_to_category_id 개수:",
    len(label_to_category_id),
)

print(
    "앞 10개:",
    list(label_to_category_id.items())[:10],
)

label_to_category_id 개수: 56
앞 10개: [(1, 1900), (2, 2483), (3, 3351), (4, 3483), (5, 3544), (6, 3743), (7, 3832), (8, 4543), (9, 12081), (10, 12247)]


In [24]:
# ============================================================
# Submission DataFrame 생성
# ============================================================

SCORE_THRESHOLD = 0.05

submission_df = predictions_to_submission(
    predictions=predictions,
    score_threshold=SCORE_THRESHOLD,
    label_to_category_id=label_to_category_id,
)

print("submission row 수:", len(submission_df))

display(submission_df.head())

submission row 수: 28507


,annotation_id,image_id,category_id,bbox_x,bbox_y,bbox_w,bbox_h,score
0,1,1,35206,148.303406,255.443787,225.915588,122.726044,0.951989
1,2,1,33880,568.121887,121.491394,329.060425,344.771576,0.339096
2,3,1,18147,575.021912,96.026222,346.773193,350.568687,0.336709
3,4,1,33880,177.260361,741.988342,184.240219,307.917053,0.308466
4,5,1,19232,185.566010,747.833191,167.607086,294.595154,0.307032


In [25]:
# ============================================================
# Submission 최종 검증
# ============================================================

print("columns:", submission_df.columns.tolist())

print(
    "annotation_id unique:",
    submission_df["annotation_id"].is_unique,
)

print(
    "image_id 수:",
    submission_df["image_id"].nunique(),
)

print(
    "category_id 범위:",
    submission_df["category_id"].min(),
    "~",
    submission_df["category_id"].max(),
)

print(
    "category_id 예시:",
    sorted(submission_df["category_id"].unique())[:10],
)

print(
    "score 범위:",
    submission_df["score"].min(),
    "~",
    submission_df["score"].max(),
)

columns: ['annotation_id', 'image_id', 'category_id', 'bbox_x', 'bbox_y', 'bbox_w', 'bbox_h', 'score']
annotation_id unique: True
image_id 수: 842
category_id 범위: 1900 ~ 41768
category_id 예시: [np.int64(1900), np.int64(2483), np.int64(3351), np.int64(3483), np.int64(3544), np.int64(3743), np.int64(3832), np.int64(4543), np.int64(12081), np.int64(12778)]
score 범위: 0.05000229924917221 ~ 0.9990949630737305


In [28]:
# ============================================================
# Submission CSV 저장
# ============================================================

save_submission(
    submission_df=submission_df,
    output_path=SUBMISSION_PATH,
)

print("저장 완료:", SUBMISSION_PATH)

저장 완료: /content/drive/MyDrive/코드잇_파트2_3팀_프로젝트/pill-object-detection/outputs/submissions/faster_rcnn_baseline_thr005.csv


In [31]:
# ============================================================
# 저장된 Submission CSV 최종 확인
# ============================================================

check_df = pd.read_csv(SUBMISSION_PATH)

print("shape:", check_df.shape)
print("columns:", check_df.columns.tolist())
print("annotation_id unique:", check_df["annotation_id"].is_unique)
print("image_id 수:", check_df["image_id"].nunique())

print(
    "category_id 범위:",
    check_df["category_id"].min(),
    "~",
    check_df["category_id"].max(),
)

print(
    "score 범위:",
    check_df["score"].min(),
    "~",
    check_df["score"].max(),
)

display(check_df.head())

shape: (28507, 8)
columns: ['annotation_id', 'image_id', 'category_id', 'bbox_x', 'bbox_y', 'bbox_w', 'bbox_h', 'score']
annotation_id unique: True
image_id 수: 842
category_id 범위: 1900 ~ 41768
score 범위: 0.0500022992491722 ~ 0.9990949630737304


,annotation_id,image_id,category_id,bbox_x,bbox_y,bbox_w,bbox_h,score
0,1,1,35206,148.303406,255.443787,225.915588,122.726044,0.951989
1,2,1,33880,568.121887,121.491394,329.060425,344.771576,0.339096
2,3,1,18147,575.021912,96.026222,346.773193,350.568687,0.336709
3,4,1,33880,177.260361,741.988342,184.240219,307.917053,0.308466
4,5,1,19232,185.566010,747.833191,167.607086,294.595154,0.307032
